In [2]:
import pandas as pd
import numpy as np
from faker import Faker
import os

fake = Faker()
rows = 2000

# إنشاء فولدر عشان نلم فيه الـ 10 ملفات
if not os.path.exists('Project_Data'):
    os.makedirs('Project_Data')

def generate_full_system():
    # قائمة بأسماء الـ 10 جداول اللي هنخلقها
    tables = ['Users', 'Products', 'Suppliers', 'Orders', 'OrderDetails', 
              'Inventory', 'Shipping', 'Payments', 'Returns', 'Staff']
    
    for table in tables:
        data = {}
        # توليد 10 أعمدة لكل جدول بشكل تلقائي
        for i in range(1, 11):
            col_name = f'Column_{i}'
            # توزيع بيانات عشوائية (أرقام، نصوص، تواريخ)
            if i == 1: data[f'{table}ID'] = range(1, rows + 1)
            elif i % 2 == 0: data[col_name] = [fake.word() for _ in range(rows)]
            elif i % 3 == 0: data[col_name] = np.random.randint(100, 1000, rows)
            else: data[col_name] = [fake.date_between(start_date='-2y') for _ in range(rows)]
            
        df = pd.DataFrame(data)
        df.to_csv(f'Project_Data/{table}_Data.csv', index=False)
        print(f"✅ تم إنشاء جدول: {table} (2000 سطر)")

generate_full_system()

✅ تم إنشاء جدول: Users (2000 سطر)
✅ تم إنشاء جدول: Products (2000 سطر)
✅ تم إنشاء جدول: Suppliers (2000 سطر)
✅ تم إنشاء جدول: Orders (2000 سطر)
✅ تم إنشاء جدول: OrderDetails (2000 سطر)
✅ تم إنشاء جدول: Inventory (2000 سطر)
✅ تم إنشاء جدول: Shipping (2000 سطر)
✅ تم إنشاء جدول: Payments (2000 سطر)
✅ تم إنشاء جدول: Returns (2000 سطر)
✅ تم إنشاء جدول: Staff (2000 سطر)


In [1]:
%pip install pandas numpy faker

  Using cached faker-40.8.0-py3-none-any.whl.metadata (16 kB)
Using cached faker-40.8.0-py3-none-any.whl (2.0 MB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd
import pyodbc
import os

# بيانات السيرفر
server = 'kerolos' 
database = 'Kero_Mega_Project'

# إنشاء الاتصال المباشر
conn_str = f'DRIVER={{SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;'
path = 'Project_Data/'
files = [f for f in os.listdir(path) if f.endswith('.csv')]

print("🚀 جاري الرفع المباشر (Direct Upload)...")

try:
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()
    
    for file in files:
        table_name = file.replace('_Data.csv', '').replace('.csv', '')
        df = pd.read_csv(os.path.join(path, file))
        
        # تحويل الداتا لنوع يقبله السيكوال
        df = df.astype(object).where(pd.notnull(df), None)
        
        # مسح الجدول لو موجود عشان نبدأ على نظافة
        cursor.execute(f"IF OBJECT_ID('{table_name}', 'U') IS NOT NULL DROP TABLE {table_name}")
        
        # إنشاء الأعمدة (كلها نصوص مؤقتاً عشان نضمن الرفع)
        cols = ", ".join([f"[{c}] NVARCHAR(MAX)" for c in df.columns])
        cursor.execute(f"CREATE TABLE {table_name} ({cols})")
        
        # حقن البيانات (2000 سطر)
        query = f"INSERT INTO {table_name} VALUES ({','.join(['?' for _ in df.columns])})"
        for row in df.values.tolist():
            cursor.execute(query, row)
            
        print(f"✅ تم رفع جدول: {table_name} بنجاح!")
    
    conn.commit()
    print("\n🎯 مبروك يا كيرو! الـ 20,000 سطر بقوا في السيكوال والجدول جاهز.")
    conn.close()

except Exception as e:
    print(f"❌ لسه فيه عكننة: {e}")

🚀 جاري الرفع المباشر (Direct Upload)...
✅ تم رفع جدول: Inventory بنجاح!
✅ تم رفع جدول: OrderDetails بنجاح!
✅ تم رفع جدول: Orders بنجاح!
✅ تم رفع جدول: Payments بنجاح!
✅ تم رفع جدول: Products بنجاح!
✅ تم رفع جدول: Returns بنجاح!
✅ تم رفع جدول: Shipping بنجاح!
✅ تم رفع جدول: Staff بنجاح!
✅ تم رفع جدول: Suppliers بنجاح!
✅ تم رفع جدول: Users بنجاح!

🎯 مبروك يا كيرو! الـ 20,000 سطر بقوا في السيكوال والجدول جاهز.


In [4]:
%pip install sqlalchemy pyodbc

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
    --------------------------------------- 0.0/2.1 MB 100.9 kB/s eta 0:00:21
    --------------------------------------- 0.0/2.1 MB 100.9 kB/s eta 0:00:21
    --------------------------------------- 0.0/2.1 MB 100.9 kB/s eta 0:00:21
    --------------------------------------- 0.0/2.1 MB 89.6 kB/s eta 0:00:24
   - -------------------------------------- 0.1/2.1 MB 126.1 k


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
